# Panduan Membangun Model Deep Learning (MLP) untuk Dataset Resep

Berikut adalah tahapan yang direkomendasikan agar model MLP yang kamu bangun memiliki akurasi optimal:

## 1. Eksplorasi dan Praproses Dataset
- Baca dataset (misal: `clean_recipes_5000.json`).
- Analisis fitur dan target.
- Bersihkan data (handle missing value, duplikasi, outlier).
- Lakukan encoding (misal: one-hot, label encoding) untuk data kategorikal.
- Normalisasi/standarisasi fitur numerik.
- Split data menjadi data latih dan data uji (misal: 80:20).

## 2. Membangun Model MLP
- Tentukan arsitektur (jumlah layer, neuron, aktivasi).
- Implementasi model dengan TensorFlow/Keras.
- Pilih loss function dan optimizer yang sesuai.

## 3. Training dan Evaluasi
- Latih model pada data latih.
- Evaluasi model pada data uji (gunakan metrik akurasi, MAE, dsb).
- Lakukan tuning hyperparameter jika perlu.

## 4. Simpan dan Inference Model
- Simpan model ke format .keras atau SavedModel.
- Buat kode inference sederhana untuk menguji model pada data baru.

---

Setiap tahap di atas sangat penting untuk memastikan hasil akurasi model optimal. Jika ingin contoh kode untuk setiap tahap, silakan minta saja!

# Main Quest & Side Quest Checklist

Tahapan pengerjaan proyek deep learning (MLP) sesuai checklist:

1. Load & Eksplorasi Data
2. Praproses Data
3. Bangun Model Deep Learning (MLP)
4. Training & Evaluasi
5. Simpan & Inference Model
6. (Opsional) Deploy API & Monitoring

---

## 1. Load & Eksplorasi Data

In [32]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
import joblib
import os

In [33]:

# Load data dari file JSON
with open('data/clean_recipes_5000.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Jika data berupa list of dict, ubah ke DataFrame
if isinstance(data, list):
    df = pd.DataFrame(data)
else:
    df = pd.DataFrame([data])

# Tampilkan 5 baris pertama
df.head()

,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps,judul_bersih,bahan_bersih,Quality Score
0,Bakso Sapi (Pakai Blender),250 gram daging sapi--35 gram es batu (sekitar...,"1) Siapkan semua bahan, cincang daging sapi ke...",939,https://cookpad.com/id/resep/3431604-bakso-sap...,sapi,bakso sapi ( pakai blender ),8,"daging sapi , es batu cmx , putih telur , tepu...",10,bakso sapi pakai blender,250 gram daging sapi 35 gram es batu sekitar 4...,0.7997
1,Chilli Tuna Puff Kilat Super Yummy,1/2 pack Kulit puff instant saya merk.Edo--Bah...,1) Langkah \n1.Tumis chili tuna chunk dengan m...,516,https://cookpad.com/id/resep/4463240-chilli-tu...,ikan,chilli tuna puff kilat super yummy,9,"pack kulit puff instant merkedo , isian , kale...",3,chilli tuna puff kilat super yummy,1 2 pack kulit puff instant saya merk edo baha...,0.4689
2,Perkedel Tahu Simple,3 buah tahu petak--1 batang daun seledri--2 si...,1) Giling halus bawang merah+bawang putih+ mer...,481,https://cookpad.com/id/resep/4337985-perkedel-...,tahu,perkedel tahu simple,8,"tahu petak , batang daun seledri , bawang puti...",5,perkedel tahu simple,3 buah tahu petak 1 batang daun seledri 2 siun...,0.4827
3,Orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera--3 buah cab...,1) Goreng tempe yg sdh di potong2 dlm minyak p...,452,https://cookpad.com/id/resep/3989566-orek-temp...,tempe,orek tempe basah bumbu ulek,12,"papan tempe potong , cabe ijo , kecap manis , ...",3,orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera 3 buah cabe...,0.4223
4,Sop Iga Sapi Enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan--Secukupn...","1) Didihkan secukupnya air, lalu masukan poton...",375,https://cookpad.com/id/resep/3310336-sop-iga-s...,sapi,sop iga sapi enaaak bangeet,22,"iga sapi , tiriskan , air didihkan utk rebusan...",4,sop iga sapi enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan secukupny...",0.3550


In [13]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Title                5000 non-null   str    
 1   Ingredients          5000 non-null   str    
 2   Steps                5000 non-null   str    
 3   Loves                5000 non-null   int64  
 4   URL                  5000 non-null   str    
 5   Category             5000 non-null   str    
 6   Title Cleaned        5000 non-null   str    
 7   Total Ingredients    5000 non-null   int64  
 8   Ingredients Cleaned  5000 non-null   str    
 9   Total Steps          5000 non-null   int64  
 10  judul_bersih         5000 non-null   str    
 11  bahan_bersih         5000 non-null   str    
 12  Quality Score        5000 non-null   float64
dtypes: float64(1), int64(3), str(9)
memory usage: 507.9 KB


Title                  0
Ingredients            0
Steps                  0
Loves                  0
URL                    0
Category               0
Title Cleaned          0
Total Ingredients      0
Ingredients Cleaned    0
Total Steps            0
judul_bersih           0
bahan_bersih           0
Quality Score          0
dtype: int64

In [14]:
df.describe()

,Loves,Total Ingredients,Total Steps,Quality Score
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,22.000000,12.507000,5.448800,0.300001
std,30.643339,4.615259,2.220172,0.068726
min,6.000000,3.000000,2.000000,0.084200
25%,9.000000,9.000000,4.000000,0.255600
50%,12.000000,12.000000,5.000000,0.293400
75%,25.000000,16.000000,7.000000,0.337675
max,939.000000,25.000000,23.000000,0.799700


## 2. Praproses Data

Tahapan praproses data meliputi:
- Menghapus duplikasi
- Menangani missing value
- Encoding fitur kategorikal
- Normalisasi fitur numerik

Contoh kode:

In [15]:
# Hapus duplikasi baris
print('Sebelum hapus duplikasi:', len(df))
df = df.drop_duplicates()
print('Setelah hapus duplikasi:', len(df))

# Tangani missing value 
df = df.dropna()

df.head()

Sebelum hapus duplikasi: 5000
Setelah hapus duplikasi: 5000


,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps,judul_bersih,bahan_bersih,Quality Score
0,Bakso Sapi (Pakai Blender),250 gram daging sapi--35 gram es batu (sekitar...,"1) Siapkan semua bahan, cincang daging sapi ke...",939,https://cookpad.com/id/resep/3431604-bakso-sap...,sapi,bakso sapi ( pakai blender ),8,"daging sapi , es batu cmx , putih telur , tepu...",10,bakso sapi pakai blender,250 gram daging sapi 35 gram es batu sekitar 4...,0.7997
1,Chilli Tuna Puff Kilat Super Yummy,1/2 pack Kulit puff instant saya merk.Edo--Bah...,1) Langkah \n1.Tumis chili tuna chunk dengan m...,516,https://cookpad.com/id/resep/4463240-chilli-tu...,ikan,chilli tuna puff kilat super yummy,9,"pack kulit puff instant merkedo , isian , kale...",3,chilli tuna puff kilat super yummy,1 2 pack kulit puff instant saya merk edo baha...,0.4689
2,Perkedel Tahu Simple,3 buah tahu petak--1 batang daun seledri--2 si...,1) Giling halus bawang merah+bawang putih+ mer...,481,https://cookpad.com/id/resep/4337985-perkedel-...,tahu,perkedel tahu simple,8,"tahu petak , batang daun seledri , bawang puti...",5,perkedel tahu simple,3 buah tahu petak 1 batang daun seledri 2 siun...,0.4827
3,Orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera--3 buah cab...,1) Goreng tempe yg sdh di potong2 dlm minyak p...,452,https://cookpad.com/id/resep/3989566-orek-temp...,tempe,orek tempe basah bumbu ulek,12,"papan tempe potong , cabe ijo , kecap manis , ...",3,orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera 3 buah cabe...,0.4223
4,Sop Iga Sapi Enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan--Secukupn...","1) Didihkan secukupnya air, lalu masukan poton...",375,https://cookpad.com/id/resep/3310336-sop-iga-s...,sapi,sop iga sapi enaaak bangeet,22,"iga sapi , tiriskan , air didihkan utk rebusan...",4,sop iga sapi enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan secukupny...",0.3550


In [21]:
# kolom yang akan dipakai sebagai fitur
fitur_kolom = ['Total Ingredients', 'Total Steps', 'Loves', 'Category']
target_kolom = 'Quality Score'

# siapkan X dan y
X = df[fitur_kolom].copy()
y = df[target_kolom].copy()

print("Bentuk X:", X.shape)
print("Bentuk y:", y.shape)

Bentuk X: (5000, 4)
Bentuk y: (5000,)


In [ ]:
# Pilih fitur dan target
fitur = ['Total Ingredients', 'Total Steps', 'Loves', 'Category']
X = df[fitur].copy()
y = df['Quality Score']

# Encode hanya kolom Category
le = LabelEncoder()
X['Category'] = le.fit_transform(X['Category'].astype(str))

# Split data (sebelum scaling)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Scaling fitur numerik (fit hanya dari data train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [25]:
# SPLIT DATA: TRAIN, VALIDATION, TEST UNTUK MLP
from sklearn.model_selection import train_test_split

# Split awal: train+val dan test (85%:15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Split train dan validation dari X_temp (train:val = 70:15 dari total data)
val_size = 0.15 / 0.85  # proporsi validation dari X_temp
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size, random_state=42)

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Train: (3500, 4)
Validation: (750, 4)
Test: (750, 4)


In [34]:
# Inisialisasi scaler
scaler = StandardScaler()

# Fit hanya pada data train, lalu transform train, val, test
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling selesai.")
print("Contoh nilai pertama train setelah scaling:", X_train_scaled[0])

Scaling selesai.
Contoh nilai pertama train setelah scaling: [ 0.35180491  3.50891434 -0.39773784  0.21315585]


In [35]:

# Asumsi X_train_scaled sudah ada
input_shape = X_train_scaled.shape[1]

inputs = tf.keras.Input(shape=(input_shape,), name='input_fitur')

# Hidden layer 1: 128 neuron
x = layers.Dense(128, activation='relu', name='hidden1')(inputs)
x = layers.Dropout(0.3, name='dropout1')(x)  

# Hidden layer 2: 64 neuron (bisa juga 32)
x = layers.Dense(64, activation='relu', name='hidden2')(x)
x = layers.Dropout(0.2, name='dropout2')(x)

# Output layer (regresi untuk Quality Score)
outputs = layers.Dense(1, activation='linear', name='output')(x)

# Model
model = Model(inputs=inputs, outputs=outputs, name='SayurKita_MLP_Quality')
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

Model: "SayurKita_MLP_Quality"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_fitur (InputLayer)        │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden1 (Dense)                 │ (None, 128)            │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,961 (35.00 KB)

 Trainable params: 8,961 (35.00 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
class RichCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1}: loss={logs['loss']:.4f}, mae={logs['mae']:.4f}, val_mae={logs.get('val_mae', 'N/A')}")
        if logs.get('val_mae', 1) < 0.05:
            self.model.save('best_model.keras')
            print("Best model saved.")
            self.model.stop_training = True

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping]
)

Epoch 1/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.4336 - mae: 0.3350 - val_loss: 0.0198 - val_mae: 0.1283
Epoch 2/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0748 - mae: 0.1406 - val_loss: 0.0120 - val_mae: 0.0955
Epoch 3/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1232 - mae: 0.1036 - val_loss: 0.0052 - val_mae: 0.0623
Epoch 4/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0180 - mae: 0.0752 - val_loss: 0.0029 - val_mae: 0.0424
Epoch 5/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0363 - mae: 0.0651 - val_loss: 0.0090 - val_mae: 0.0555
Epoch 6/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0096 - mae: 0.0622 - val_loss: 0.0039 - val_mae: 0.0508
Epoch 7/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0067 - mae: 0.0530 - val_loss: 0.0026 - val_mae: 0.0390
Epoch 8/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0048 - mae: 0.0461 - val_loss: 0.0023 - val_mae: 0.0363
Epoch 9/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/